In [5]:
from sklearn.datasets import fetch_openml
import pandas as pd
from pathlib import Path

In [6]:
# Set python path to applications/ml_coding/xgboost
project_root = Path.cwd().parent
print(f"Project root set to: {project_root}")
import sys
sys.path.insert(0, str(project_root))

Project root set to: /Users/jerry/projects/ai_apps/applications/ml_coding/xgboost


In [2]:
cache_dir = "/tmp/sklearn_data/jerryjcw"

In [7]:
from src.ranking.data import download_movielens_100k

dataset = download_movielens_100k(cache_dir=cache_dir)

In [26]:
# Run ls on the dataset
import os 
from src.ranking.config import MOVIE_COLUMNS, RATINGS_COLUMNS

df_ratings = pd.read_csv(Path(dataset) / "u.data", names=RATINGS_COLUMNS, sep="\t")
df_users = pd.read_csv(Path(dataset) / "u.item", names=MOVIE_COLUMNS, sep="|", encoding="latin-1")

In [36]:
df_ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [32]:
# Remove users with less than 20 ratings
user_counts = df_ratings["user_id"].value_counts()
users_to_keep = user_counts[user_counts >= 20].index
df_ratings = df_ratings[df_ratings["user_id"].isin(users_to_keep)]

# Print number of users removed and number of users kept
num_users_removed = len(user_counts) - len(users_to_keep)
num_users_kept = len(users_to_keep)
print(f"Removed {num_users_removed} users with less than 20 ratings.")

Removed 0 users with less than 20 ratings.


In [100]:
# Split user's records into train, val and test sets, with a split of 60%, 20% and 20% respectively
from sklearn.model_selection import train_test_split

def split_user_records(df, user_id_col="user_id", test_size=0.2, val_size=0.2, random_state=42):
    train_records = []
    val_records = []
    test_records = []
    
    for user_id, user_df in df.groupby(user_id_col):
        train_val_df, test_df = train_test_split(user_df, test_size=test_size, random_state=random_state)
        train_df, val_df = train_test_split(train_val_df, test_size=val_size/(1-test_size), random_state=random_state)
        
        train_records.append(train_df)
        val_records.append(val_df)
        test_records.append(test_df)
    
    return pd.concat(train_records), pd.concat(val_records), pd.concat(test_records)


df_overall = df_ratings.merge(df_users, on="movie_id", how="left")
train_df, val_df, test_df = split_user_records(df_overall)

user_stats = train_df.groupby("user_id")["rating"].agg(["mean", "std"]).reset_index()
movie_stats = train_df.groupby("movie_id")["rating"].agg(["mean", "std"]).reset_index()
global_mean, global_std = train_df["rating"].mean(), train_df["rating"].std()

# Fill the na values with zeros
user_stats = user_stats.fillna(0)
movie_stats = movie_stats.fillna(0)

def post_process(df):
    df = df.merge(user_stats, on="user_id", how="left", suffixes=("", "_user"))
    df = df.merge(movie_stats, on="movie_id", how="left", suffixes=("", "_movie"))
    df = df.drop(columns=["movie_id", "title", "release_date", "video_release_date", "imdb_url"])
    df = df.fillna(0)
    df = df.reset_index(drop=True)
    return df

train_df = post_process(train_df)
val_df = post_process(val_df)
test_df = post_process(test_df)

In [102]:
# re-arrange the train_df, val_df, test_df to have the form of [features-without-rating-and-userid, rating, user_id]
import numpy as np

feature_cols = [col for col in train_df.columns if col not in ["rating", "user_id"]]
train_df  = train_df[feature_cols + ["rating", "user_id"]]
val_df = val_df[feature_cols + ["rating", "user_id"]]
test_df = test_df[feature_cols + ["rating", "user_id"]]

train_features_df, train_ratings, train_qids = train_df.drop(columns=["rating", "user_id"]), train_df["rating"].astype(np.float32), train_df["user_id"].astype(np.int32).to_numpy()
val_features_df, val_ratings, val_qids = val_df.drop(columns=["rating", "user_id"]), val_df["rating"].astype(np.float32), val_df["user_id"].astype(np.int32).to_numpy()
test_features_df, test_ratings, test_qids = test_df.drop(columns=["rating", "user_id"]), test_df["rating"].astype(np.float32), test_df["user_id"].astype(np.int32).to_numpy()


In [103]:
len(train_features_df), len(train_ratings), len(train_qids)

(59262, 59262, 59262)

In [104]:
len(val_features_df), len(val_ratings), len(val_qids)

(20128, 20128, 20128)

In [105]:
# Create a xgboost ranker
import xgboost as xgb
from src.ranking.config import DEFAULT_XGB_PARAMS

default_params = DEFAULT_XGB_PARAMS
default_params["early_stopping_rounds"] = 40
 
ranker = xgb.XGBRanker(**default_params)
ranker.fit(train_features_df, train_ratings, qid=train_qids, eval_set=[(val_features_df, val_ratings)], eval_qid=[val_qids], verbose=True)



[0]	validation_0-ndcg@5:0.76098	validation_0-ndcg@10:0.80702
[1]	validation_0-ndcg@5:0.76302	validation_0-ndcg@10:0.80902
[2]	validation_0-ndcg@5:0.76348	validation_0-ndcg@10:0.80892
[3]	validation_0-ndcg@5:0.76510	validation_0-ndcg@10:0.80939
[4]	validation_0-ndcg@5:0.76438	validation_0-ndcg@10:0.80855
[5]	validation_0-ndcg@5:0.76553	validation_0-ndcg@10:0.80855
[6]	validation_0-ndcg@5:0.76550	validation_0-ndcg@10:0.80921
[7]	validation_0-ndcg@5:0.76527	validation_0-ndcg@10:0.80846
[8]	validation_0-ndcg@5:0.76535	validation_0-ndcg@10:0.80845
[9]	validation_0-ndcg@5:0.76444	validation_0-ndcg@10:0.80893
[10]	validation_0-ndcg@5:0.76505	validation_0-ndcg@10:0.80940
[11]	validation_0-ndcg@5:0.76568	validation_0-ndcg@10:0.81002
[12]	validation_0-ndcg@5:0.76587	validation_0-ndcg@10:0.80936
[13]	validation_0-ndcg@5:0.76623	validation_0-ndcg@10:0.81012
[14]	validation_0-ndcg@5:0.76678	validation_0-ndcg@10:0.81010
[15]	validation_0-ndcg@5:0.76670	validation_0-ndcg@10:0.80986
[16]	validation_0-

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'rank:ndcg'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",40
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from skle

In [106]:
# Define ndcg@k metric
def _dcg_at_k(scores, labels, k):
    arg_scores = np.argsort(scores)[::-1][:k]
    labels_at_k = labels[arg_scores]

    dcg = np.sum(labels_at_k / np.log2(np.arange(2, np.size(arg_scores) + 2)))
    return dcg

def ndcg_at_k(scores, labels, k):
    dcg = _dcg_at_k(scores, labels, k)
    ideal_dcg = _dcg_at_k(labels, labels, k)
    return dcg / ideal_dcg if ideal_dcg > 0 else 0.0


def mrr_at_k(scores, labels, k, relevant_threshold=4):
    arg_scores = np.argsort(scores)[::-1][:k]
    if max(labels) < relevant_threshold:
        return 0.0
    for idx in arg_scores:
        if labels[idx] > relevant_threshold:
            return 1.0 / (idx + 1)
    return 0.0


# Compute the scores for the test set
def compute_scores(model, features_df, scores_df, qids):
    scores = model.predict(features_df)
    result_df = pd.DataFrame({"user_id": qids, "score": scores, "labels": scores_df.to_numpy().tolist()})
    # Group by user_id and collect scores and labels into lists
    result_df = result_df.groupby("user_id").apply(lambda x: pd.Series({
        "score": x["score"].to_numpy(),
        "labels": x["labels"].to_numpy()
    })).reset_index()

    # For each user, compute the ndcg@k 
    for k in [5, 10, 20]:
        result_df[f"ndcg@{k}"] = result_df.apply(lambda x: ndcg_at_k(x["score"], x["labels"], k), axis=1)
    return result_df

test_scores = compute_scores(ranker, val_features_df, val_ratings, val_qids)

In [107]:
test_scores["ndcg@5"].mean(), test_scores["ndcg@10"].mean(), test_scores["ndcg@20"].mean()

(np.float64(0.8843892525972112),
 np.float64(0.9079799852076401),
 np.float64(0.9284050537882638))

In [109]:
# Use parameter tuning to find the best parameters for the xgboost ranker
from sklearn.model_selection import GridSearchCV
param_grid = {
    "max_depth": [2, 3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "min_child_weight": [0.1, 0.5, 1, 2, 5, 10],
    "n_estimators": [100, 200, 300, 500],
}

# For each combination of parameters, train a xgboost ranker, and compute the ndcg@10 on the validation set. 
best_score = 0.0
best_params = None

# Use iterotools.product to iterate over all combinations of parameters
import itertools

for max_depth, learning_rate, min_child_weight, n_estimators in itertools.product(param_grid["max_depth"], param_grid["learning_rate"], param_grid["min_child_weight"], param_grid["n_estimators"]):
    variant = {
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "min_child_weight": min_child_weight,
        "n_estimators": n_estimators,
        "early_stopping_rounds": 40,
    }
    params = default_params.copy()
    params.update(variant)

    ranker = xgb.XGBRanker(**params)
    ranker.fit(train_features_df, train_ratings, qid=train_qids, eval_set=[(val_features_df, val_ratings)], eval_qid=[val_qids], verbose=False)
    test_scores = compute_scores(ranker, val_features_df, val_ratings, val_qids)
    score = test_scores["ndcg@10"].mean()
    if score > best_score:
        print(f"New best score: {score} with params: {params}")
        best_score = score
        best_params = params

print(f"Best score: {best_score} with params: {best_params}")

New best score: 0.9078392111523742 with params: {'objective': 'rank:ndcg', 'eval_metric': ['ndcg@5', 'ndcg@10'], 'lambdarank_pair_method': 'topk', 'lambdarank_num_pair_per_sample': 8, 'tree_method': 'hist', 'n_estimators': 100, 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 0.1, 'gamma': 0.0, 'subsample': 0.9, 'colsample_bytree': 0.9, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'random_state': 42, 'n_jobs': -1, 'early_stopping_rounds': 40}
New best score: 0.9090733647478884 with params: {'objective': 'rank:ndcg', 'eval_metric': ['ndcg@5', 'ndcg@10'], 'lambdarank_pair_method': 'topk', 'lambdarank_num_pair_per_sample': 8, 'tree_method': 'hist', 'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 2, 'min_child_weight': 0.1, 'gamma': 0.0, 'subsample': 0.9, 'colsample_bytree': 0.9, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'random_state': 42, 'n_jobs': -1, 'early_stopping_rounds': 40}
New best score: 0.9094348351514832 with params: {'objective': 'rank:ndcg', 'eval_metric': ['ndcg@5', '

In [110]:
# Now update min_child_weight, gamma, subsample and colsample_bytree parameters

parameter_grid = {
    "gamma": [0, 0.1, 0.2, 0.5],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
}

print(f"Best params from previous tuning: {best_params}")
print(f"Best score from previous tuning: {best_score}")
for gamma, subsample, colsample_bytree in itertools.product(parameter_grid["gamma"], parameter_grid["subsample"], parameter_grid["colsample_bytree"]):
    variant = {
        "gamma": gamma,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "early_stopping_rounds": 40,
    }
    params = best_params.copy()
    params.update(variant)

    ranker = xgb.XGBRanker(**params)
    ranker.fit(train_features_df, train_ratings, qid=train_qids, eval_set=[(val_features_df, val_ratings)], eval_qid=[val_qids], verbose=False)
    test_scores = compute_scores(ranker, val_features_df, val_ratings, val_qids)
    score = test_scores["ndcg@10"].mean()
    if score > best_score:
        print(f"New best score: {score} with params: {params}")
        best_score = score
        best_params = params 

print(f"Best params from tuning: {best_params}")
print(f"Best score from tuning: {best_score}")

Best params from previous tuning: {'objective': 'rank:ndcg', 'eval_metric': ['ndcg@5', 'ndcg@10'], 'lambdarank_pair_method': 'topk', 'lambdarank_num_pair_per_sample': 8, 'tree_method': 'hist', 'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 2, 'min_child_weight': 0.1, 'gamma': 0.0, 'subsample': 0.9, 'colsample_bytree': 0.9, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'random_state': 42, 'n_jobs': -1, 'early_stopping_rounds': 40}
Best score from previous tuning: 0.9097364502717246
New best score: 0.909834512232703 with params: {'objective': 'rank:ndcg', 'eval_metric': ['ndcg@5', 'ndcg@10'], 'lambdarank_pair_method': 'topk', 'lambdarank_num_pair_per_sample': 8, 'tree_method': 'hist', 'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 2, 'min_child_weight': 0.1, 'gamma': 0, 'subsample': 0.7, 'colsample_bytree': 1.0, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'random_state': 42, 'n_jobs': -1, 'early_stopping_rounds': 40}
Best params from tuning: {'objective': 'rank:ndcg', 'eval_metric': ['

In [111]:
# Check regularization parameters
parameter_grid = {
    "reg_alpha": [0, 0.1, 0.2, 0.5, 1, 2, 5, 10],
    "reg_lambda": [0, 0.1, 0.2, 0.5, 1, 2, 5, 10],
}   

print(f"Best params from previous tuning: {best_params}")
print(f"Best score from previous tuning: {best_score}")
for reg_alpha, reg_lambda in itertools.product(parameter_grid["reg_alpha"], parameter_grid["reg_lambda"]):
    variant = {
        "reg_alpha": reg_alpha,
        "reg_lambda": reg_lambda,
        "early_stopping_rounds": 40,
    }
    params = best_params.copy()
    params.update(variant)

    ranker = xgb.XGBRanker(**params)
    ranker.fit(train_features_df, train_ratings, qid=train_qids, eval_set=[(val_features_df, val_ratings)], eval_qid=[val_qids], verbose=False)
    test_scores = compute_scores(ranker, val_features_df, val_ratings, val_qids)
    score = test_scores["ndcg@10"].mean()
    if score > best_score:
        print(f"New best score: {score} with params: {params}")
        best_score = score
        best_params = params

print(f"Best params from tuning: {best_params}")
print(f"Best score from tuning: {best_score}")

Best params from previous tuning: {'objective': 'rank:ndcg', 'eval_metric': ['ndcg@5', 'ndcg@10'], 'lambdarank_pair_method': 'topk', 'lambdarank_num_pair_per_sample': 8, 'tree_method': 'hist', 'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 2, 'min_child_weight': 0.1, 'gamma': 0, 'subsample': 0.7, 'colsample_bytree': 1.0, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'random_state': 42, 'n_jobs': -1, 'early_stopping_rounds': 40}
Best score from previous tuning: 0.909834512232703
New best score: 0.909864414925742 with params: {'objective': 'rank:ndcg', 'eval_metric': ['ndcg@5', 'ndcg@10'], 'lambdarank_pair_method': 'topk', 'lambdarank_num_pair_per_sample': 8, 'tree_method': 'hist', 'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 2, 'min_child_weight': 0.1, 'gamma': 0, 'subsample': 0.7, 'colsample_bytree': 1.0, 'reg_alpha': 0.1, 'reg_lambda': 0, 'random_state': 42, 'n_jobs': -1, 'early_stopping_rounds': 40}
New best score: 0.9098731756873181 with params: {'objective': 'rank:ndcg',